In [1]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------------ --------------------- 5.8/12.8 MB 43.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 42.5 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import glob
import pandas as pd
import nltk
from nltk import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
import wordcloud

In [4]:
# modeli za ustrezno obdelavo stavkov, besed, ločil....
nltk.download('punkt')     # stavki, besede
nltk.download('wordnet') #lemmatizacija
nltk.download('averaged_perceptron_tagger') #POS tagganje
nltk.download('omw-1.4') 
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\mokro\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_percep

True

In [40]:
# tokenization and lemmatization 
lemmatizer= WordNetLemmatizer()

In [41]:
# pokupčkamo besede s podobnim korenom, pomenom skupaj
# run, runs, running -> run
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('RB'):
        return wordnet.ADV
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    else:
        return wordnet.NOUN

In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")

In [6]:
def tokenize_lematize(tekst):
    doc = nlp(tekst)
    
    izbrane_besede = []
    
    # želimo odstraniti (osebe, kraji, jeziki, narodi)
    odstrani_entitete = {'PERSON', 'GPE', 'LOC', 'NORP', 'FAC', 'ORG'}
    
    # ustvarimo množico besed, ki so del prepoznanih entitet (imen)
    imena_v_tekstu = {ent.text.lower() for ent in doc.ents if ent.label_ in odstrani_entitete}

    for token in doc:
        # če je beseda ime (NER)
        if token.text.lower() in imena_v_tekstu:
            continue
            
        # spaCy uporablja oznake: NOUN ADJ
        if token.pos_ in ['NOUN', 'ADJ']:
            beseda = token.lemma_.lower()
            
            # samo črke in dolžina nad 2 znaka
            if beseda.isalpha() and len(beseda) > 2:
                izbrane_besede.append(beseda)
                
    return izbrane_besede

In [7]:
import random
from sklearn.feature_extraction import text

In [8]:
base_vectorizer = CountVectorizer(stop_words='english')
base_stopwords = base_vectorizer.get_stop_words()


custom_words = {
    # založniški šum 
    'book', 'novel', 'story', 'author', 'literature', 'edition', 'seller', 
    'read', 'reader', 'page', 'chapter', 'write', 'writer', 'publish', 
    'publication', 'review', 'times', 'york', 'print', 'copy', 'best', 
    'original', 'classic', 'introduction', 'note', 'debut', 'thriller',
    'unique', 'fascinating', 'scandal', 'major',
    
    # splošni opisi 
    'way', 'thing', 'important', 'practical', 'young', 'boy', 'girl', 
    'human', 'people', 'great', 'good', 'bad', 'true', 'new', 'old',
    'life', 'world', 'everything', 'day', 'time', 'year', 'make',
    'take', 'come', 'think', 'feel', 'know', 'look', 'want', 'large', 'small',
    'man', 'woman', 'literary', 'secret', 
    
    #iz izpisa
    'professional', 'guide', 'experience', 'natural', 'vivid', 'narrative',
    'compelling', 'extraordinary', 'powerful', 'voice', 'mind'
}


all_stopwords = list(base_stopwords.union(custom_words))

In [24]:
# CountVectorizer odstrani 'stopwords' in ustvari nenegativno matriko, na (i, j)-tem mestu
# imamo pojavitev besede i v j-tem dokumentu (glej zapiske na tablici)


# vzamemo 49/50 knjig, eno bomo potem poskusali uvrstiti med žanre
filepaths = glob.glob(r'C:\Users\mokro\Desktop\diploma\leto_2003\03_ang_opisi\*.txt')[:150]
# min_df=2, max_df=0.9 odstranita redke in pogoste besede, to uniči celoten rezultat
vectorizer= CountVectorizer(stop_words=all_stopwords, 
                            tokenizer= tokenize_lematize,
                            input = 'filename', 
                            encoding='latin-1', 
                            min_df=3, 
                            max_df=10)

In [25]:
X = vectorizer.fit_transform(filepaths) 


c:\Users\mokro\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [26]:
print(X)

# malo lepše, prikaz
dense_matrix = X.toarray()
print(dense_matrix)

#še lepše
feature_names = vectorizer.get_feature_names_out()
df = pd.DataFrame(dense_matrix, columns=feature_names)
print(df.head())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 2411 stored elements and shape (150, 531)>
  Coords	Values
  (0, 159)	1
  (0, 385)	1
  (0, 316)	1
  (0, 332)	1
  (0, 394)	1
  (0, 492)	1
  (0, 358)	1
  (0, 204)	1
  (0, 182)	2
  (0, 449)	1
  (0, 98)	1
  (0, 447)	1
  (0, 1)	1
  (0, 106)	1
  (0, 472)	1
  (0, 115)	1
  (0, 348)	1
  (0, 277)	1
  (1, 55)	1
  (1, 67)	1
  (1, 188)	1
  (1, 6)	1
  (1, 488)	1
  (1, 58)	1
  (1, 263)	1
  :	:
  (147, 188)	1
  (147, 516)	3
  (147, 198)	1
  (147, 114)	1
  (147, 80)	1
  (147, 63)	3
  (147, 494)	1
  (147, 229)	1
  (147, 317)	1
  (147, 50)	1
  (147, 262)	1
  (147, 312)	1
  (147, 426)	1
  (148, 316)	1
  (148, 332)	1
  (148, 355)	1
  (148, 68)	1
  (149, 98)	1
  (149, 215)	1
  (149, 309)	1
  (149, 402)	1
  (149, 110)	1
  (149, 530)	2
  (149, 53)	1
  (149, 170)	2
[[0 1 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 2]]
   account  action  actual  addition  adolescence  affair  agent  ali

In [27]:
from sklearn.decomposition import PCA

In [28]:
pca = PCA(n_components=4)  
X_pca = pca.fit_transform(X)

In [29]:
print(pca.explained_variance_ratio_)
print("Skupaj:", sum(pca.explained_variance_ratio_))

[0.03354035 0.03045698 0.02681899 0.02613718]
Skupaj: 0.11695349903311984


In [30]:
feature_names = vectorizer.get_feature_names_out()

for i, comp in enumerate(pca.components_):
    top_words = [feature_names[j] for j in comp.argsort()[-15:]]
    print(f"PC{i+1}: {top_words}")

PC1: ['country', 'historical', 'camp', 'belief', 'leader', 'savage', 'church', 'bone', 'realm', 'violence', 'brother', 'violent', 'religious', 'fundamentalist', 'faith']
PC2: ['ambition', 'art', 'slave', 'early', 'land', 'era', 'imagination', 'bloody', 'fiction', 'revolution', 'epic', 'empire', 'country', 'camp', 'historical']
PC3: ['cost', 'joy', 'connection', 'common', 'information', 'safe', 'play', 'role', 'ancient', 'master', 'medical', 'care', 'doctor', 'pain', 'birth']
PC4: ['evil', 'economic', 'memoir', 'understanding', 'horror', 'truth', 'law', 'quest', 'birth', 'epic', 'historical', 'resource', 'revolution', 'country', 'camp']


In [31]:
# import matplotlib.pyplot as plt
# labels = X_pca.argmax(axis=1)   

# pairs = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]

# for i, j in pairs:
#     plt.figure()
#     plt.scatter(
#         X_pca[:, i],
#         X_pca[:, j],
#         c=labels,       
#         cmap="tab10"    
#     )
#     plt.xlabel(f"PC{i+1}")
#     plt.ylabel(f"PC{j+1}")
#     plt.title(f"PC{i+1} vs PC{j+1}")
#     plt.colorbar(label="žanr")   
#     plt.show()

# plt.show()